In [541]:
import pandas as pd
from pandas import DataFrame,Series
import numpy as np

### Hierarchical Indexing ###

In [542]:
data = pd.Series(np.random.uniform(size=9),
                 index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])
data

a  1    0.896438
   2    0.615358
   3    0.929937
b  1    0.564980
   3    0.309158
c  1    0.542631
   2    0.883035
d  2    0.595821
   3    0.017363
dtype: float64

In [543]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

In [544]:
data.b
data["b":"c"]
data.loc[["b","c"]]

b  1    0.564980
   3    0.309158
c  1    0.542631
   2    0.883035
dtype: float64

In [545]:
data.loc[:,2]

a    0.615358
c    0.883035
d    0.595821
dtype: float64

In [546]:
data.unstack()

,1,2,3
a,0.896438,0.615358,0.929937
b,0.564980,NaN,0.309158
c,0.542631,0.883035,NaN
d,NaN,0.595821,0.017363


In [547]:
data.unstack().stack()

a  1    0.896438
   2    0.615358
   3    0.929937
b  1    0.564980
   2         NaN
   3    0.309158
c  1    0.542631
   2    0.883035
   3         NaN
d  1         NaN
   2    0.595821
   3    0.017363
dtype: float64

In [548]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                        index=[["a", "a", "b", "b"], [1, 2, 1, 2]],
                        columns=[["Ohio", "Ohio", "Colorado"],
                        ["Green", "Red", "Green"]])
frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

In [549]:
frame.index.names = ["key1","key2"]
frame

Ohio     Colorado
          Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [550]:
frame.columns.names = ["state","colour"]
frame

state      Ohio     Colorado
colour    Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [551]:
frame.index.nlevels

2

In [552]:
frame["Ohio"]

colour     Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

In [553]:
pd.MultiIndex.from_arrays([["Ohio" , "Ohio" , "Colorado"],
                           ["Green" , "Red" , "Green"]],
                           names=["state" , "color"])

MultiIndex([(    'Ohio', 'Green'),
            (    'Ohio',   'Red'),
            ('Colorado', 'Green')],
           names=['state', 'color'])

Reordering and Sorting Levels

In [554]:
frame.swaplevel("key1","key2")

state      Ohio     Colorado
colour    Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

In [555]:
frame.sort_index(level=1)

frame.swaplevel(0,1).sort_index(level=0)

frame.sort_index

<bound method DataFrame.sort_index of state      Ohio     Colorado
colour    Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11>

Summary Statistics by Level

In [556]:
frame.groupby(level="key2").sum()

state   Ohio     Colorado
colour Green Red    Green
key2                     
1          6   8       10
2         12  14       16

Indexing with a DataFrame’s columns

In [557]:
frame = pd.DataFrame({"a": range(7), "b": range(7, 0, -1),
                    "c": ["one", "one", "one", "two", "two",
                    "two", "two"],
                    "d": [0, 1, 2, 0, 1, 2, 3]})
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


In [558]:
frame2 = frame.set_index(["c","d"])
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

In [559]:
frame2.reset_index()

,c,d,a,b
0,one,0,0,7
1,one,1,1,6
2,one,2,2,5
3,two,0,3,4
4,two,1,4,3
5,two,2,5,2
6,two,3,6,1


### Combining and Merging Datasets ###

Database-Style DataFrame Joins

In [560]:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "a", "b"],
                    "data1": pd.Series(range(7), dtype="Int64")})
df2 = pd.DataFrame({"key": ["a", "b", "d"],
                    "data2": pd.Series(range(3), dtype="Int64")})
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


In [561]:
df2

,key,data2
0,a,0
1,b,1
2,d,2


In [562]:
#NATURAL JOIN (NATURAL MEANING NOT SPECIFYING WHICH COLUMN WE ARE JOINING HERE)
pd.merge(df1,df2)

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


In [563]:
#EQUI JOIN
pd.merge(df1,df2,on="key")

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


In [564]:
df3 = pd.DataFrame({"lkey": ["b", "b", "a", "c", "a", "a", "b"],
                    "data1": pd.Series(range(7), dtype="Int64")})
df4 = pd.DataFrame({"rkey": ["a", "b", "d"],
                    "data2": pd.Series(range(3), dtype="Int64")})

In [565]:
pd.merge(df3,df4,left_on="lkey",right_on="rkey")

,lkey,data1,rkey,data2
0,b,0,b,1
1,b,1,b,1
2,a,2,a,0
3,a,4,a,0
4,a,5,a,0
5,b,6,b,1


In [566]:
#FULL OUTER JOIN
pd.merge(df1,df2,how="outer")

,key,data1,data2
0,a,2,0
1,a,4,0
2,a,5,0
3,b,0,1
4,b,1,1
5,b,6,1
6,c,3,<NA>
7,d,<NA>,2


In [567]:
pd.merge(df3,df4,left_on="lkey",right_on='rkey',how="outer")

,lkey,data1,rkey,data2
0,a,2,a,0
1,a,4,a,0
2,a,5,a,0
3,b,0,b,1
4,b,1,b,1
5,b,6,b,1
6,c,3,NaN,<NA>
7,NaN,<NA>,d,2


In [568]:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"],
                    "data1": pd.Series(range(6), dtype="Int64")})
df2 = pd.DataFrame({"key": ["a", "b", "a", "b", "d"],
                    "data2": pd.Series(range(5), dtype="Int64")})
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [569]:
df2

,key,data2
0,a,0
1,b,1
2,a,2
3,b,3
4,d,4


In [570]:
#LEFT JOIN
pd.merge(df1,df2,on="key",how="left")

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,c,3,<NA>
7,a,4,0
8,a,4,2
9,b,5,1


In [571]:
#INNER JOIN
pd.merge(df1, df2, how="inner")

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,a,4,0
7,a,4,2
8,b,5,1
9,b,5,3


In [572]:
left = pd.DataFrame({"key1": ["foo", "foo", "bar"],
                     "key2": ["one", "two", "one"],
                     "lval": pd.Series([1, 2, 3], dtype='Int64')})
right = pd.DataFrame({"key1": ["foo", "foo", "bar", "bar"],
                      "key2": ["one", "one", "one", "two"],
                      "rval": pd.Series([4, 5, 6, 7], dtype='Int64')})
left

,key1,key2,lval
0,foo,one,1
1,foo,two,2
2,bar,one,3


In [573]:
right

,key1,key2,rval
0,foo,one,4
1,foo,one,5
2,bar,one,6
3,bar,two,7


In [574]:
pd.merge(left,right,on=["key1","key2"], how="outer").set_index(["key1","key2"]).reset_index()

,key1,key2,lval,rval
0,bar,one,3,6
1,bar,two,<NA>,7
2,foo,one,1,4
3,foo,one,1,5
4,foo,two,2,<NA>


In [575]:
pd.merge(left,right,on="key1",suffixes=(["_left","_right"]))

,key1,key2_left,lval,key2_right,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


Merging on Index

In [576]:
left1 = pd.DataFrame({"key":["a","b","a","a","b","c"],
                      "value": pd.Series(range(6), dtype="Int64")})
right1 = pd.DataFrame({"group_val":[3.5,7]}, index = ["a","b"])
left1

,key,value
0,a,0
1,b,1
2,a,2
3,a,3
4,b,4
5,c,5


In [577]:
right1

,group_val
a,3.5
b,7.0


In [578]:
pd.merge(left1,right1,left_on="key",right_index=True)

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0


In [579]:
pd.merge(left1,right1,left_on="key",right_index=True,how="outer")

,key,value,group_val
0,a,0,3.5
2,a,2,3.5
3,a,3,3.5
1,b,1,7.0
4,b,4,7.0
5,c,5,NaN


In [580]:
lefth = pd.DataFrame({"key1": ["Ohio", "Ohio", "Ohio","Nevada", "Nevada"],\
                      "key2": [2000, 2001, 2002, 2001, 2002],
                      "data": pd.Series(range(5), dtype="Int64")})

righth_index = pd.MultiIndex.from_arrays(
    [
        ["Nevada", "Nevada", "Ohio", "Ohio", "Ohio", "Ohio"],
        [2001, 2000, 2000, 2000, 2001, 2002]
    ]
)

righth = pd.DataFrame({"event1": pd.Series([0, 2, 4, 6, 8, 10], dtype="Int64",
                                            index=righth_index),
                        "event2": pd.Series([1, 3, 5, 7, 9, 11], dtype="Int64",
                                            index=righth_index)})

lefth.sort_index()

,key1,key2,data
0,Ohio,2000,0
1,Ohio,2001,1
2,Ohio,2002,2
3,Nevada,2001,3
4,Nevada,2002,4


In [581]:
righth.sort_index()

event1  event2
Nevada 2000       2       3
       2001       0       1
Ohio   2000       4       5
       2000       6       7
       2001       8       9
       2002      10      11

In [582]:
pd.merge(lefth,righth,left_on=["key1","key2"],right_index=True).sort_index()

,key1,key2,data,event1,event2
0,Ohio,2000,0,4,5
0,Ohio,2000,0,6,7
1,Ohio,2001,1,8,9
2,Ohio,2002,2,10,11
3,Nevada,2001,3,0,1


In [583]:
pd.merge(lefth,righth,left_on=["key1","key2"],how="outer",right_index=True).sort_index()

,key1,key2,data,event1,event2
0,Ohio,2000,0,4,5
0,Ohio,2000,0,6,7
1,Ohio,2001,1,8,9
2,Ohio,2002,2,10,11
3,Nevada,2001,3,0,1
4,Nevada,2000,<NA>,2,3
4,Nevada,2002,4,<NA>,<NA>


In [584]:
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]],
                     index=["a", "c", "e"],
                     columns=["Ohio", "Nevada"]).astype("Int64")

right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                      index=["b", "c", "d", "e"],
                      columns=["Missouri", "Alabama"]).astype("Int64")

left2

,Ohio,Nevada
a,1,2
c,3,4
e,5,6


In [585]:
right2

,Missouri,Alabama
b,7,8
c,9,10
d,11,12
e,13,14


In [586]:
pd.merge(left2,right2,how="outer",left_index=True,right_index=True)

,Ohio,Nevada,Missouri,Alabama
a,1,2,<NA>,<NA>
b,<NA>,<NA>,7,8
c,3,4,9,10
d,<NA>,<NA>,11,12
e,5,6,13,14


JOIN

In [587]:
#JUST LIKE ABOVE
left2.join(right2,how="outer") # NOTE: IN JOIN APNE AAP HI INDEX IDENTIFY HOJATA HAI

,Ohio,Nevada,Missouri,Alabama
a,1,2,<NA>,<NA>
b,<NA>,<NA>,7,8
c,3,4,9,10
d,<NA>,<NA>,11,12
e,5,6,13,14


In [588]:
#JUST LIKE ABOVE EXAMPLES
left1.join(right1,on="key")

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0
5,c,5,NaN


In [589]:
another = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [16., 17.]],
                       index=["a", "c", "e", "f"],
                       columns=["New York", "Oregon"])

another

,New York,Oregon
a,7.0,8.0
c,9.0,10.0
e,11.0,12.0
f,16.0,17.0


In [590]:
left2.join([right2,another])

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1,2,<NA>,<NA>,7.0,8.0
c,3,4,9,10,9.0,10.0
e,5,6,13,14,11.0,12.0


In [591]:
left2.join([right2,another], how="outer")

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1,2,<NA>,<NA>,7.0,8.0
b,<NA>,<NA>,7,8,NaN,NaN
c,3,4,9,10,9.0,10.0
d,<NA>,<NA>,11,12,NaN,NaN
e,5,6,13,14,11.0,12.0
f,<NA>,<NA>,<NA>,<NA>,16.0,17.0


Concatenating Along an Axis

In [592]:
arr = np.arange(12).reshape((3,4))
arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [593]:
np.concatenate([arr,arr],axis=1)

array([[ 0,  1,  2,  3,  0,  1,  2,  3],
       [ 4,  5,  6,  7,  4,  5,  6,  7],
       [ 8,  9, 10, 11,  8,  9, 10, 11]])

In [594]:
s1 = pd.Series([0, 1], index=["a", "b"], dtype="Int64")
s2 = pd.Series([2, 3, 4], index=["c", "d", "e"], dtype="Int64")
s3 = pd.Series([5, 6], index=["f", "g"], dtype="Int64")
s1

a    0
b    1
dtype: Int64

In [595]:
s2

c    2
d    3
e    4
dtype: Int64

In [596]:
s3

f    5
g    6
dtype: Int64

In [597]:
pd.concat([s1,s2,s3],axis=1) #ITS IS OUTER JOIN DEFAULT

,0,1,2
a,0,<NA>,<NA>
b,1,<NA>,<NA>
c,<NA>,2,<NA>
d,<NA>,3,<NA>
e,<NA>,4,<NA>
f,<NA>,<NA>,5
g,<NA>,<NA>,6


In [598]:
s4 = pd.concat([s1,s3],axis=0)
s4

a    0
b    1
f    5
g    6
dtype: Int64

In [599]:
pd.concat([s1,s4],axis=1)

,0,1
a,0,0
b,1,1
f,<NA>,5
g,<NA>,6


In [600]:
pd.concat([s1,s4],axis=1,join="inner")

,0,1
a,0,0
b,1,1


In [601]:
result = pd.concat([s1, s1, s3], keys=["one", "two", "three"])
result

one    a    0
       b    1
two    a    0
       b    1
three  f    5
       g    6
dtype: Int64

In [602]:
result.unstack()

,a,b,f,g
one,0,1,<NA>,<NA>
two,0,1,<NA>,<NA>
three,<NA>,<NA>,5,6


In [603]:
pd.concat([s1, s1, s3],axis=1, keys=["one", "two", "three"])

,one,two,three
a,0,0,<NA>
b,1,1,<NA>
f,<NA>,<NA>,5
g,<NA>,<NA>,6


In [604]:
pd.concat([s1, s2, s3], axis="columns", keys=["one", "two", "three"])

,one,two,three
a,0,<NA>,<NA>
b,1,<NA>,<NA>
c,<NA>,2,<NA>
d,<NA>,3,<NA>
e,<NA>,4,<NA>
f,<NA>,<NA>,5
g,<NA>,<NA>,6


In [605]:
df1 = pd.DataFrame(np.arange(6).reshape((3,2)), index=["a","b","c"],
                   columns=["one","two"])
df2 = pd.DataFrame(5+np.arange(4).reshape(2,2), index=["a","c"],
                   columns = ["three","four"])
df1

,one,two
a,0,1
b,2,3
c,4,5


In [606]:
df2

,three,four
a,5,6
c,7,8


In [607]:
pd.concat([df1,df2], axis = 1, keys = ["level1","level2"])

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

In [608]:
pd.concat({"level1" : df1,
           "level2" : df2}, axis = 1)

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

In [609]:
pd.concat({"level1" : df1,
           "level2" : df2}, axis = 1, names = ["upper","lower"])

upper level1     level2     
lower    one two  three four
a          0   1    5.0  6.0
b          2   3    NaN  NaN
c          4   5    7.0  8.0

In [610]:
df1 = pd.DataFrame(np.random.standard_normal((3, 4)),
                   columns=["a", "b", "c", "d"])
df2 = pd.DataFrame(np.random.standard_normal((2, 3)),
                   columns=["b", "d", "a"])
df1

,a,b,c,d
0,-1.700490,-0.537683,1.641193,0.797165
1,0.467004,-1.754978,1.055233,-0.262866
2,0.138851,0.433034,1.113572,-1.107817


In [611]:
df2

,b,d,a
0,-1.764080,-1.330717,0.965221
1,0.454121,-0.022995,-0.812551


In [612]:
pd.concat([df1,df2],ignore_index=True)

,a,b,c,d
0,-1.700490,-0.537683,1.641193,0.797165
1,0.467004,-1.754978,1.055233,-0.262866
2,0.138851,0.433034,1.113572,-1.107817
3,0.965221,-1.764080,NaN,-1.330717
4,-0.812551,0.454121,NaN,-0.022995


Combining Data with Overlap

In [613]:
a = pd.Series([np.nan, 2.5, 0.0, 3.5, 4.5, np.nan],
              index=["f", "e", "d", "c", "b", "a"])
b = pd.Series([0., np.nan, 2., np.nan, np.nan, 5.],
              index=["a", "b", "c", "d", "e", "f"])
a

f    NaN
e    2.5
d    0.0
c    3.5
b    4.5
a    NaN
dtype: float64

In [614]:
b

a    0.0
b    NaN
c    2.0
d    NaN
e    NaN
f    5.0
dtype: float64

In [615]:
np.where(pd.isna(a),b,a)

array([0. , 2.5, 0. , 3.5, 4.5, 5. ])

In [616]:
a.combine_first(b)

a    0.0
b    4.5
c    3.5
d    0.0
e    2.5
f    5.0
dtype: float64

In [617]:
df1 = pd.DataFrame({"a":[1.,np.nan,5.,np.nan],
                    "b":[np.nan,2.,np.nan,6.],
                    "c":range(2,18,4)})
df1

,a,b,c
0,1.0,NaN,2
1,NaN,2.0,6
2,5.0,NaN,10
3,NaN,6.0,14


In [618]:
df2 = pd.DataFrame({"a":[5.,4.,np.nan,3.,7.],
                    "b":[np.nan,3.,4.,6.,8.]})
df2

,a,b
0,5.0,NaN
1,4.0,3.0
2,NaN,4.0
3,3.0,6.0
4,7.0,8.0


In [619]:
df1.combine_first(df2)

,a,b,c
0,1.0,NaN,2.0
1,4.0,2.0,6.0
2,5.0,4.0,10.0
3,3.0,6.0,14.0
4,7.0,8.0,NaN


### Reshaping and Pivoting ###

Reshaping with Hierarchical Indexing

In [620]:
data = pd.DataFrame(np.arange(6).reshape((2,3)),
                    index= pd.Index(["Ohio","Colorado"], name="state"),
                    columns= pd.Index(["one","two","three"],name="number")
                    )
data

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


In [621]:
result = data.stack()
result

state     number
Ohio      one       0
          two       1
          three     2
Colorado  one       3
          two       4
          three     5
dtype: int64

In [622]:
result.unstack()

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


In [623]:
result.unstack(level=0)

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


In [624]:
result.unstack(level="state")

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


In [625]:
s1 = pd.Series([0, 1, 2, 3], index=["a", "b", "c", "d"], dtype="Int64")
s2 = pd.Series([4, 5, 6], index=["c", "d", "e"], dtype="Int64")
data2 = pd.concat([s1, s2], keys=["one", "two"])
data2

one  a    0
     b    1
     c    2
     d    3
two  c    4
     d    5
     e    6
dtype: Int64

In [626]:
data2.unstack()

,a,b,c,d,e
one,0,1,2,3,<NA>
two,<NA>,<NA>,4,5,6


In [627]:
data2.unstack(level=0)

,one,two
a,0,<NA>
b,1,<NA>
c,2,4
d,3,5
e,<NA>,6


In [628]:
data2.unstack().stack()

one  a       0
     b       1
     c       2
     d       3
     e    <NA>
two  a    <NA>
     b    <NA>
     c       4
     d       5
     e       6
dtype: Int64

In [629]:
data2.unstack().stack().dropna()

one  a    0
     b    1
     c    2
     d    3
two  c    4
     d    5
     e    6
dtype: Int64

In [630]:
result

state     number
Ohio      one       0
          two       1
          three     2
Colorado  one       3
          two       4
          three     5
dtype: int64

In [631]:
df = pd.DataFrame({"left" : result, "right" : result + 5},
                  columns = pd.Index(["left","right"],name="side"))
df

side             left  right
state    number             
Ohio     one        0      5
         two        1      6
         three      2      7
Colorado one        3      8
         two        4      9
         three      5     10

In [632]:
df.unstack(level=0)

side   left          right         
state  Ohio Colorado  Ohio Colorado
number                             
one       0        3     5        8
two       1        4     6        9
three     2        5     7       10

In [633]:
df.unstack(level=0).stack(level="side")

state         Ohio  Colorado
number side                 
one    left      0         3
       right     5         8
two    left      1         4
       right     6         9
three  left      2         5
       right     7        10

Pivoting “Long” to “Wide” Format

In [634]:
data = pd.read_csv("examples/macrodata.csv")
data = data.loc[:,["year", "quarter", "realgdp", "infl", "unemp"]]
data

,year,quarter,realgdp,infl,unemp
0,1959,1,2710.349,0.00,5.8
1,1959,2,2778.801,2.34,5.1
2,1959,3,2775.488,2.74,5.3
3,1959,4,2785.204,0.27,5.6
4,1960,1,2847.699,2.31,5.2
...,...,...,...,...,...
198,2008,3,13324.600,-3.16,6.0
199,2008,4,13141.920,-8.79,6.9
200,2009,1,12925.410,0.94,8.1
201,2009,2,12901.504,3.37,9.2


In [635]:
periods = pd.PeriodIndex.from_fields(
    year=data.pop("year"),
    quarter=data.pop("quarter"),
    freq="Q"
)
periods.name = "date"
periods

PeriodIndex(['1959Q1', '1959Q2', '1959Q3', '1959Q4', '1960Q1', '1960Q2',
             '1960Q3', '1960Q4', '1961Q1', '1961Q2',
             ...
             '2007Q2', '2007Q3', '2007Q4', '2008Q1', '2008Q2', '2008Q3',
             '2008Q4', '2009Q1', '2009Q2', '2009Q3'],
            dtype='period[Q-DEC]', name='date', length=203)

In [636]:
data.index = periods.to_timestamp("D")
data.columns.name = "item"
data.head()

item,realgdp,infl,unemp
date,,,
1959-01-01,2710.349,0.00,5.8
1959-04-01,2778.801,2.34,5.1
1959-07-01,2775.488,2.74,5.3
1959-10-01,2785.204,0.27,5.6
1960-01-01,2847.699,2.31,5.2


In [640]:
long_data = (data.stack()
             .reset_index(name="value")
             )
long_data.head(10)

,date,item,value
0,1959-01-01,realgdp,2710.349
1,1959-01-01,infl,0.000
2,1959-01-01,unemp,5.800
3,1959-04-01,realgdp,2778.801
4,1959-04-01,infl,2.340
5,1959-04-01,unemp,5.100
6,1959-07-01,realgdp,2775.488
7,1959-07-01,infl,2.740
8,1959-07-01,unemp,5.300
9,1959-10-01,realgdp,2785.204


In [643]:
pivoted = long_data.pivot(index="date", columns="item",
                          values="value")
pivoted.head()

item,infl,realgdp,unemp
date,,,
1959-01-01,0.00,2710.349,5.8
1959-04-01,2.34,2778.801,5.1
1959-07-01,2.74,2775.488,5.3
1959-10-01,0.27,2785.204,5.6
1960-01-01,2.31,2847.699,5.2


In [656]:
long_data["value2"] = np.random.standard_normal(len(long_data))
long_data[:10]

,date,item,value,value2
0,1959-01-01,realgdp,2710.349,-1.163926
1,1959-01-01,infl,0.000,-2.104955
2,1959-01-01,unemp,5.800,-0.244743
3,1959-04-01,realgdp,2778.801,-0.339731
4,1959-04-01,infl,2.340,-1.202122
5,1959-04-01,unemp,5.100,0.164725
6,1959-07-01,realgdp,2775.488,-0.137823
7,1959-07-01,infl,2.740,0.113945
8,1959-07-01,unemp,5.300,0.729866
9,1959-10-01,realgdp,2785.204,-0.261212


In [669]:
pivoted = long_data.pivot(index="date", columns="item")
pivoted.head()

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-01-01  0.00  2710.349   5.8 -2.104955 -1.163926 -0.244743
1959-04-01  2.34  2778.801   5.1 -1.202122 -0.339731  0.164725
1959-07-01  2.74  2775.488   5.3  0.113945 -0.137823  0.729866
1959-10-01  0.27  2785.204   5.6 -1.655292 -0.261212 -0.247738
1960-01-01  2.31  2847.699   5.2  2.654677 -0.953899  0.498721

In [672]:
unstacked = long_data.set_index(["date", "item"]).unstack(level="item")
unstacked.head()    

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-01-01  0.00  2710.349   5.8 -2.104955 -1.163926 -0.244743
1959-04-01  2.34  2778.801   5.1 -1.202122 -0.339731  0.164725
1959-07-01  2.74  2775.488   5.3  0.113945 -0.137823  0.729866
1959-10-01  0.27  2785.204   5.6 -1.655292 -0.261212 -0.247738
1960-01-01  2.31  2847.699   5.2  2.654677 -0.953899  0.498721

Pivoting “Wide” to “Long” Format


Study from the book or while doing the project